# Recover Mixture Embeddings (Submodule-Local)

This notebook is self-contained inside `chemprop/examples/MixtureDesign`.
It uses local copied weights/data and local runtime helpers under `runtime/`.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "chemprop/examples/MixtureDesign").exists():
            return p
    raise RuntimeError(f"Could not locate repo root from {start}")


repo_root = find_repo_root(Path.cwd())
mixture_dir = repo_root / "chemprop/examples/MixtureDesign"
runtime_dir = mixture_dir / "runtime"
if str(runtime_dir) not in sys.path:
    sys.path.insert(0, str(runtime_dir))

import train_gnn_ffn as tgm
from utils.mixtures import collate_mixture

print(f"mixture_dir: {mixture_dir}")


mixture_dir: /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign


/Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign/runtime/utils/mixtures.py:110: SyntaxWarning: invalid escape sequence '\s'
  """whether any :class:`MolGraph`\s are stored in this batch"""
/Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign/runtime/utils/mixtures.py:276: SyntaxWarning: invalid escape sequence '\m'
  Include a bit for all hydrogen bond numbers in the interval :math:`[1, \mathtt{max\_hbond\_num}]`
/Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign/runtime/utils/mixtures.py:393: SyntaxWarning: invalid escape sequence '\s'
  """A :class:`BatchMixtureGraph` represents a batch of individual :class:`MixtureGraph`\s.


In [2]:
mix_csv = mixture_dir / "data/dcn_mix_exp.csv"
model_paths = [mixture_dir / "weights/gsolv_mix_comp_mixture_context_solute_10k_fold_00_model.pt"]
output_dir = mixture_dir / "outputs/gsolv_mix_comp_to_dcn_mix"
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(mix_csv)
all_data = tgm.build_all_data(df.reset_index(drop=True), target_col="value", solute_component_index=-1)
n_components = len(all_data) - 1
dset = tgm.build_mixture_dataset(all_data, n_components=n_components, use_mixmp=True)
loader = DataLoader(dset, batch_size=128, shuffle=False, collate_fn=collate_mixture)

y = pd.to_numeric(df["value"], errors="coerce").fillna(df["value"].mean()).to_numpy(dtype=float).reshape(-1, 1)
dummy_scaler = StandardScaler().fit(y)

rows = []
for model_path in model_paths:
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    use_mixmp = ckpt.get("mixmp") is not None

    model = tgm.build_model(
        n_components=n_components,
        scaler=dummy_scaler,
        aggregation="weightedsum",
        solute_component_index=-1,
        use_mixmp=use_mixmp,
        x_d_dim=0,
    )
    _ = model.message_passing.load_state_dict(ckpt["message_passing"], strict=False)
    _ = model.agg.load_state_dict(ckpt["mixagg"], strict=False)

    predictor_state = dict(ckpt["predictor"])
    w0 = predictor_state["ffn.0.0.weight"]
    d_mix = model.agg.output_dim
    if w0.shape[1] == 2 * d_mix and model.predictor.ffn[0][0].weight.shape[1] == d_mix:
        predictor_state["ffn.0.0.weight"] = w0[:, -d_mix:].clone()
    model.predictor.load_state_dict(predictor_state, strict=True)
    model.eval()

    preds, mix_embs = [], []
    with torch.no_grad():
        for batch in loader:
            bmgs, v_ds, x_d_batch, *_ = batch
            z = model.fingerprint(bmgs, v_ds, x_d_batch)
            y_hat = model.predictor(z)
            preds.append(y_hat.cpu().numpy().reshape(-1, 1))
            mix_embs.append(z.cpu().numpy())

    pred = np.concatenate(preds, axis=0).reshape(-1)
    emb = np.concatenate(mix_embs, axis=0)
    stem = model_path.stem
    np.savez(output_dir / f"{stem}_embeddings.npz", prediction=pred, mixture_embedding=emb, row_index=np.arange(len(df), dtype=int))
    pd.DataFrame(emb, columns=[f"mix_emb_{i}" for i in range(emb.shape[1])]).to_csv(output_dir / f"{stem}_mixture_embeddings.csv", index=False)
    rows.append({"model": model_path.name, "n_rows": len(df), "mix_emb_dim": int(emb.shape[1]), "pred_mean": float(np.mean(pred))})

summary_df = pd.DataFrame(rows)
summary_df.to_csv(output_dir / "embedding_export_summary.csv", index=False)
print(summary_df)


                                               model  n_rows  mix_emb_dim  \
0  gsolv_mix_comp_mixture_context_solute_10k_fold...     484          101   

   pred_mean  
0  -9.969319  
